In [1]:
from datasets import load_dataset
import json
path="/Users/charon/Desktop/VLFFD-main/output.json"
dataset = load_dataset("json", data_files=path, split="train")
next(iter(dataset))

{'question': 'Can we determine the authenticity of the face presented in this image?',
 'imagepath': '/Users/charon/Documents/FaceForensics++/original_sequences/youtube/c23/frames/000/344.png',
 'answer': 'real',
 'reason': 'This is a real face. The image displays a relatively symmetrical facial structure with features consistent with human anatomy. The skin texture appears natural, showing subtle variations in color and tone, indicating natural skin characteristics, such as pores and fine wrinkles. The lighting seems even, with gradual transitions between light and shadow areas. There are no apparent blending artifacts or distortions within the features like eyes, nose and mouth, and there are no indications of texture abnormalities. The overall impression is one of photographic realism, lending credibility to its genuineness.'}

In [2]:
reasoning_start = "<thinking>"
reasoning_end   = "</thinking>"
solution_start = "<answer>"
solution_end = "</answer>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your answer between {solution_start}{solution_end},solution should be 'real' or 'fake'
"""

from PIL import Image
def process_example(example):
    example["prompt"]=[
        # {
        
        # "role": "system",
        # "content":[{"type": "text", "text": system_prompt}]
        # },
        {
            "role":"user",
            "content":[
                {"type":"image"},
                {"type":"text","text":example["question"]},
            ]
        }
    ]
    example["completion"] = [
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": reasoning_start + example["reason"] + reasoning_end+'\n'+ solution_start + example["answer"] + solution_end},
            ]
        }
    ]

    return example

dataset = dataset.map(process_example)

dataset[0]

{'question': 'Can we determine the authenticity of the face presented in this image?',
 'imagepath': '/Users/charon/Documents/FaceForensics++/original_sequences/youtube/c23/frames/000/344.png',
 'answer': 'real',
 'reason': 'This is a real face. The image displays a relatively symmetrical facial structure with features consistent with human anatomy. The skin texture appears natural, showing subtle variations in color and tone, indicating natural skin characteristics, such as pores and fine wrinkles. The lighting seems even, with gradual transitions between light and shadow areas. There are no apparent blending artifacts or distortions within the features like eyes, nose and mouth, and there are no indications of texture abnormalities. The overall impression is one of photographic realism, lending credibility to its genuineness.',
 'prompt': [{'content': [{'text': None, 'type': 'image'},
    {'text': 'Can we determine the authenticity of the face presented in this image?',
     'type': 

In [3]:
from transformers import AutoProcessor,Gemma3ForConditionalGeneration
from PIL import Image
import requests
import torch
model_id="google/gemma-3-4b-it"
model=Gemma3ForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto",
    low_cpu_mem_usage=True
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
processor=AutoProcessor.from_pretrained(model_id)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
processor.pad_token_id=processor.tokenizer.eos_token_id
processor.padding_side = "left"
processor.tokenizer.padding_side = "left"

In [6]:
from peft import get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,        # 任务类型：因果语言模型
    inference_mode=False,                # 训练模式
    r=8,                                 # LoRA秩 (低秩矩阵维度)
    lora_alpha=32,                       # Alpha参数 (缩放因子)
    lora_dropout=0.05,                   # Dropout概率
    target_modules=["q_proj", "v_proj"],  # 目标模块 (根据模型结构调整)
    bias="none",                         # 不添加偏置
    modules_to_save=["lm_head"],         # 额外保存的模块
)
# 3. 应用PEFT配置到基础模型
model = get_peft_model(model, peft_config)

# 4. 打印可训练参数比例
model.print_trainable_parameters()


/Users/charon/miniconda3/envs/llm/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
trainable params: 674,476,032 || all params: 4,974,555,504 || trainable%: 13.5585


In [7]:
import re

match_format = re.compile(
    rf"^[\s]{{0,}}"\
    rf"{reasoning_start}.+?{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

In [8]:
from bert_score import score

def reward_func(completions,**kwargs):
    rewards = []
    print("==== COMPLETION ====")
    print(completions)
    # 从 kwargs 提取字段
    answers = kwargs.get("answer", [])
    reasons = kwargs.get("reason", [])
    for comp, ans, gt_reason in zip(completions, answers, reasons):
         # 尝试提取 Answer
        pred_ans_match = re.search(r"Answer:\s*(.*?)\n", comp)
        pred_ans = pred_ans_match.group(1).strip() if pred_ans_match else ""

        # 尝试提取 Reason
        pred_reason_match = re.search(r"Reason:\s*(.*?)(</end_of_turn>|$)", comp, re.DOTALL)
        pred_reason = pred_reason_match.group(1).strip() if pred_reason_match else ""

        # 正确性奖励
        correct_score = 2.0 if pred_ans == ans else -1.0

        # 推理质量奖励
        _, _, bert_F = score([pred_reason], [gt_reason], lang="en")
        reason_score = bert_F.item() * 1.5

          # 格式奖励
        format_ok = 1.0 if ("Answer:" in comp) and ("Reason:" in comp) else 0.0


        rewards.append(0.6*correct_score + 0.3*reason_score + 0.1*format_ok)

    return rewards

In [9]:
max_seq_length = 2048
max_prompt_length = 1024
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_torch_fused",
    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 300,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

INFO 06-04 19:31:11 [__init__.py:239] Automatically detected platform cpu.


In [10]:
from transformers.image_utils import make_nested_list_of_images,make_flat_list_of_images

In [11]:
img1=Image.open("/Users/charon/Desktop/llm/llm/image.png").convert("RGB")
img2=Image.open("/Users/charon/Desktop/llm/llm/image.png").convert("RGB")
images=[]
images.append([img1])
images.append([img2])
print(images)
batched_images = make_nested_list_of_images(images)
print(batched_images)
batch_num_crops = [[22 for _ in range(len(images))] for images in batched_images]
print(batch_num_crops)

[[<PIL.Image.Image image mode=RGB size=1337x421 at 0x327B36E30>], [<PIL.Image.Image image mode=RGB size=1337x421 at 0x12033F100>]]
[[<PIL.Image.Image image mode=RGB size=1337x421 at 0x327B36E30>], [<PIL.Image.Image image mode=RGB size=1337x421 at 0x12033F100>]]
[[22], [22]]


In [12]:
from typing import Any
from PIL import Image

def collate_fn(examples: list[dict[str, Any]]):
    texts = []
    images = []
    for example in examples:
        img=Image.open(example["imagepath"]).convert("RGB")
        images.append([img])
        texts.append(processor.apply_chat_template(
            example["prompt"], add_generation_prompt=False, tokenize=False
        ).strip())
    if len(images) != len(texts):
        print(f"警告: 图像数({len(images)})与文本数({len(texts)})不一致")
        # 截断较长的列表以匹配较短的列表
        min_length = min(len(images), len(texts))
        images = images[:min_length]
        texts = texts[:min_length]
    # Tokenize the texts and process the images
    print("----------------before processor----------------")
    print(f"Texts: {texts}")

    print(f"Images: {images}")
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    print("---------------------------")
    # The labels are the input_ids, with the padding and image tokens masked in
    # the loss computation
    labels = batch["input_ids"].clone()

    # Mask image tokens
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    # Mask tokens that are not used in the loss computation
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch

In [ ]:
class CustomGRPOTrainer(GRPOTrainer):
    def __init__(self, *args, data_collator=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.data_collator = data_collator

# 使用自定义训练器
trainer = CustomGRPOTrainer(
    model=model,
    # processing_class=processor,
    reward_funcs=[match_format_exactly, reward_func],
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn  # 现在可以传递自定义collate_fn了
)
trainer.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/Users/charon/miniconda3/envs/llm/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


----------------before processor----------------
Texts: ['<bos><start_of_turn>user\n<start_of_image>Can you verify the authenticity of this facial image?<end_of_turn>', '<bos><start_of_turn>user\n<start_of_image>Can you verify the authenticity of this facial image?<end_of_turn>', '<bos><start_of_turn>user\n<start_of_image>Is there any evidence of image tampering here?<end_of_turn>', '<bos><start_of_turn>user\n<start_of_image>Is there any evidence of image tampering here?<end_of_turn>']
Images: [[<PIL.Image.Image image mode=RGB size=256x256 at 0x583BE7580>], [<PIL.Image.Image image mode=RGB size=256x256 at 0x592913940>], [<PIL.Image.Image image mode=RGB size=256x256 at 0x592912170>], [<PIL.Image.Image image mode=RGB size=256x256 at 0x592910A00>]]
---------------------------
----------------before processor----------------
Texts: ['<bos><start_of_turn>user\n<start_of_image>Can you verify the authenticity of this facial image?<end_of_turn>', '<bos><start_of_turn>user\n<start_of_image>Can 

TypeError: string indices must be integers

: 